<img src=https://courseware.decoded.com/images/decoded/logo-decoded.png align=left width=100px/>

# M09 Capstone project - AutoML template
---
<small>© 2026 Decoded Limited. All rights reserved. Website: https://decoded.com</small>

## Welcome to the end-to-end low-code Machine learning workflow

In this notebook, you will build a complete machine-learning pipeline from **EDA through model building to hyperparameter tuning** — all using **PyCaret** for rapid prototyping.

### What you will learn

| Stage | Tool | What you do |
| ----- | ---- | ----------- |
| 1. EDA | `ydata-profiling` | Generate interactive reports, check data quality |
| 2. Data prep | `pandas` / `scikit-learn` | Clean, split, and prepare data |
| 3. Baseline | PyCaret `setup()` | Initialise the training environment |
| 4. Compare | PyCaret `compare_models()` | Test 15+ algorithms automatically |
| 5. Tune | PyCaret `tune_model()` | Optimise hyperparameters with grid search |
| 6. Deploy | PyCaret `save_model()` | Package the final pipeline for production |

### Datasets
- **Juice (Classification):** Predict customer beverage brand preference (Citrus Hill vs Minute Maid).
- **Insurance (Regression):** Predict medical charges based on patient attributes.

## 0. Install dependencies and import libraries

In [54]:
# Install dependencies (uncomment if running for the first time)
# !pip install ydata-profiling pycaret

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# Import the rapid EDA library
from data_profiling import ProfileReport
# Get the sample dataset
from pycaret.datasets import get_data
# get all classification models
from pycaret.classification import *
# get all plots for classification performance evaluation
from pycaret.plots.classification import confusion_matrix, roc_curve
# get all regression models
from pycaret.regression import *
# get all plots for regression performance evaluation
from pycaret.plots.regression import residuals
# get feature importance plot
from pycaret.plots.feature import permutation_importance
# perform train-test split
from sklearn.model_selection import train_test_split

print("All libraries imported successfully.")

All libraries imported successfully.


---
## Activity A: Classification EDA — Juice Dataset

In [5]:
# Load the Juice classification dataset
juice_data = get_data("juice", verbose=False)

# Quick sanity check
print("Shape:", juice_data.shape)
print("Target:", juice_data["Purchase"].value_counts().to_dict())
juice_data.head()

Shape: (1070, 19)
Target: {'CH': 653, 'MM': 417}


,Id,Purchase,WeekofPurchase,StoreID,PriceCH,PriceMM,DiscCH,DiscMM,SpecialCH,SpecialMM,LoyalCH,SalePriceMM,SalePriceCH,PriceDiff,Store7,PctDiscMM,PctDiscCH,ListPriceDiff,STORE
0,1,CH,237,1,1.75,1.99,0.00,0.0,0,0,0.500000,1.99,1.75,0.24,No,0.000000,0.000000,0.24,1
1,2,CH,239,1,1.75,1.99,0.00,0.3,0,1,0.600000,1.69,1.75,-0.06,No,0.150754,0.000000,0.24,1
2,3,CH,245,1,1.86,2.09,0.17,0.0,0,0,0.680000,2.09,1.69,0.40,No,0.000000,0.091398,0.23,1
3,4,MM,227,1,1.69,1.69,0.00,0.0,0,0,0.400000,1.69,1.69,0.00,No,0.000000,0.000000,0.00,1
4,5,CH,228,7,1.69,1.69,0.00,0.0,0,0,0.956535,1.69,1.69,0.00,Yes,0.000000,0.000000,0.00,0


In [10]:
# Generate the rapid EDA profile report
juice_profile = ProfileReport(juice_data, title="Juice Dataset EDA")
juice_profile.to_file("juice_eda_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 19/19 [00:00<00:00, 292984.47it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

### EDA checklist (answer these before modelling)
- What is the target variable? → `Purchase` (binary: CH vs MM)
- Are there missing values? → Check the profile report
- Are there constant or near-constant columns? → Flag them
- Are numeric features on different scales? → PyCaret handles this automatically
- Are there highly correlated features? → Note them for later

---
## Activity B: Regression EDA — Insurance Dataset

In [11]:
# Load the Insurance regression dataset
insurance_data = get_data("insurance", verbose=False)

print("Shape:", insurance_data.shape)
insurance_data.head()

Shape: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [12]:
# Generate the rapid EDA profile report
insurance_profile = ProfileReport(insurance_data, title="Insurance Dataset EDA", explorative=True)
insurance_profile.to_file("insurance_eda_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 7/7 [00:00<00:00, 1378.47it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

### EDA checklist
- What is the target variable? → `charges` (continuous)
- Check for skewness in numeric features
- Note any outliers or unusual distributions
- Identify categorical vs numeric columns

---
## 1. Data Preparation

In [14]:
# --- Classification data prep ---
clf_data = juice_data.copy()
# Drop ID and redundant columns noted during EDA
clf_data.drop(columns=["Id", "DiscCH", "DiscMM", "SalePriceMM"], inplace=True, errors="ignore")
print("Classification data shape:", clf_data.shape)

# --- Regression data prep ---
reg_data = insurance_data.copy()
print("Regression data shape:", reg_data.shape)

# Split both datasets (80/20 train/test)
clf_train, clf_test = train_test_split(clf_data, test_size=0.2, random_state=42, stratify=clf_data["Purchase"])
reg_train, reg_test = train_test_split(reg_data, test_size=0.2, random_state=42)

print("Clf train:", clf_train.shape, "| Clf test:", clf_test.shape)
print("Reg train:", reg_train.shape, "| Reg test:", reg_test.shape)

Classification data shape: (1070, 15)
Regression data shape: (1338, 7)
Clf train: (856, 15) | Clf test: (214, 15)
Reg train: (1070, 7) | Reg test: (268, 7)


---
## 2. PyCaret Classification Workflow

In [15]:
# Step 1: Setup the classification environment
# Set up classification experiment in pycaret 4.0

exp = ClassificationExperiment(
    target="Purchase",
    session_id=42,
    train_size=0.7,
    fold=5,
    n_jobs=1,
    normalize=True,
    transformation=True,
).fit(clf_data)


print("Classification setup complete.")

Classification setup complete.


In [64]:
# Step 2: Compare models (finds the best algorithm)
clf_result = exp.compare_models(sort="F1", n_select=1, turbo=True)
# Display the leaderboard of models
clf_result.leaderboard


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,ridge,0.8197,0.0000,0.7635,0.7713,0.7669,0.6200,0.6205
1,lda,0.8171,0.8842,0.7533,0.7718,0.7621,0.6136,0.6140
2,lr,0.8184,0.8844,0.7395,0.7825,0.7598,0.6140,0.6152
3,catboost,0.8104,0.8746,0.7499,0.7605,0.7550,0.6004,0.6006
4,ada,0.8064,0.8715,0.7328,0.7637,0.7468,0.5903,0.5917
5,gbc,0.7984,0.8703,0.7191,0.7536,0.7351,0.5727,0.5739
6,qda,0.7770,0.8383,0.7364,0.7065,0.7209,0.5354,0.5359
7,dt,0.7744,0.7642,0.7021,0.7156,0.7076,0.5242,0.5253
8,nb,0.7583,0.8291,0.7463,0.6708,0.7060,0.5018,0.5045
9,rf,0.7757,0.8438,0.6778,0.7304,0.7024,0.5228,0.5244


In [ ]:
# Store the best model for further tuning
clf_best = clf_result.best

In [ ]:
# Step 3: Tune the best model
clf_tuned = exp.tune_model(clf_best, n_iter=20, optimize="F1")

# display the tuned model's performance using the cross validation results
clf_tuned.metrics


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold 0,0.7933,0.0,0.7069,0.7455,0.7257,0.5601,0.5606
Fold 1,0.8533,0.0,0.8276,0.8000,0.8136,0.6927,0.6930
Fold 2,0.8400,0.0,0.8136,0.7869,0.8000,0.6667,0.6670
Fold 3,0.8133,0.0,0.7797,0.7541,0.7667,0.6112,0.6114
Fold 4,0.7987,0.0,0.7069,0.7593,0.7321,0.5712,0.5721
Mean,0.8197,0.0,0.7669,0.7691,0.7676,0.6204,0.6208
Std,0.0233,0.0,0.0514,0.0207,0.0352,0.0520,0.0518


In [33]:

# Step 4: Finalise on full training data
clf_final = exp.finalize_model(clf_tuned.pipeline)

print("Tuned model:", clf_final.pipeline)

Tuned model: Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('numerical_pipeline',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('transformer',
                                                                   PowerTransformer(standardize=False)),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['WeekofPurchase', 'StoreID',
                                                   'PriceCH', 'PriceMM',
                                                   'SpecialCH', 'SpecialMM',
                                                   'LoyalCH', 'SalePriceCH',
                                                   'PriceDiff'

In [36]:
# Step 5: Evaluate on the holdout test set
clf_predictions = exp.predict_model(clf_final.pipeline)

predictions_df = clf_predictions.predictions
#display the first few rows of the predictions
predictions_df.head()

,WeekofPurchase,StoreID,PriceCH,PriceMM,SpecialCH,SpecialMM,LoyalCH,SalePriceCH,PriceDiff,Store7,PctDiscMM,PctDiscCH,ListPriceDiff,STORE,Purchase,prediction_label
426,262,4,1.99,2.09,0,0,0.104858,1.99,0.10,No,0.000000,0.000000,0.10,4,CH,1
590,253,7,1.86,2.09,0,0,0.584000,1.76,0.33,Yes,0.000000,0.053763,0.23,0,CH,0
573,255,3,1.99,2.29,0,0,0.199771,1.99,0.30,No,0.000000,0.000000,0.30,3,CH,1
789,275,2,1.96,2.18,0,1,0.600000,1.96,-0.58,No,0.366972,0.000000,0.22,2,MM,1
966,236,1,1.75,1.99,0,0,0.548160,1.75,0.24,No,0.000000,0.000000,0.24,1,MM,0


In [48]:
# Step 6: Visualise performance
fig_cm = confusion_matrix(clf_final.pipeline, exp.X_test, exp.y_test)
fig_cm.show()

In [49]:
# Create the ROC curve
fig_roc = roc_curve(clf_final.pipeline, exp.X_test, exp.y_test)
fig_roc.show()

In [57]:
# Show the feature importance plot 

# Map the target variable to numeric values for permutation importance
y_test_encoded = exp.y_test.map({'CH': 0, 'MM': 1})
# Show the feature importance plot using permutation importance
fig_fi = permutation_importance(clf_final.pipeline,X=exp.X_test,y=y_test_encoded)
fig_fi.show()


---
## 3. PyCaret Regression Workflow

In [59]:
# Step 1: Setup the regression environment
reg_exp = RegressionExperiment(
    target="charges", 
    train_size=0.7, 
    session_id=42, 
    fold=5, 
    n_jobs=1, 
    normalize=True, 
    transformation=True).fit(reg_data)

print("Regression setup complete.")

Regression setup complete.


In [66]:
# Step 2: Compare models
reg_result = reg_exp.compare_models(sort="R2", turbo=True )

reg_result.leaderboard

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,gbr,2668.8716,2.255826e+07,4723.3132,0.8461,0.4506,0.3161
1,catboost,2873.2178,2.458286e+07,4930.9750,0.8325,0.4822,0.3501
2,rf,2871.6591,2.535405e+07,5014.3010,0.8266,0.4666,0.3375
3,et,2870.2060,2.770906e+07,5246.3781,0.8094,0.4811,0.3337
4,ada,4042.6726,2.806655e+07,5270.4347,0.8056,0.5970,0.6493
5,lr,4364.7052,3.865750e+07,6205.5850,0.7350,0.6092,0.4545
6,ridge,4377.6332,3.866802e+07,6206.2001,0.7350,0.6061,0.4569
7,lar,4364.7052,3.865750e+07,6205.5850,0.7350,0.6092,0.4545
8,llar,4364.7232,3.865635e+07,6205.4839,0.7350,0.6075,0.4545
9,br,4369.8571,3.866041e+07,6205.7292,0.7350,0.6081,0.4554


In [68]:
# Store the best model for further tuning
reg_best = reg_result.best

In [69]:
# Step 3: Tune the best model
reg_tuned = reg_exp.tune_model(reg_best, optimize="R2")

# display the tuned model's performance using the cross validation results
reg_tuned.metrics


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold 0,3134.9398,3.008463e+07,5484.9461,0.8295,0.5083,0.3633
Fold 1,2820.1336,2.303480e+07,4799.4579,0.8159,0.4581,0.3011
Fold 2,2728.2140,2.647796e+07,5145.6735,0.8310,0.4332,0.3139
Fold 3,2562.0558,1.613890e+07,4017.3255,0.8826,0.4266,0.3522
Fold 4,2620.0341,1.946320e+07,4411.7121,0.8558,0.4705,0.3064
Mean,2773.0755,2.303990e+07,4771.8230,0.8430,0.4594,0.3274
Std,201.5558,4.936903e+06,519.2338,0.0236,0.0293,0.0254


In [70]:

# Step 4: Finalise on full training data
reg_final = reg_exp.finalize_model(reg_tuned.pipeline)

print("Tuned model:", reg_final.pipeline)

Tuned model: Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('numerical_pipeline',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('transformer',
                                                                   PowerTransformer(standardize=False)),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'bmi', 'children']),
                                                 ('categorical_pipeline',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                             

In [71]:
# Step 5: Evaluate on holdout set
# Step 5: Evaluate on the holdout test set
reg_predictions = reg_exp.predict_model(reg_final.pipeline)

reg_predictions_df = reg_predictions.predictions
#display the first few rows of the predictions
reg_predictions_df.head()

,age,sex,bmi,children,smoker,region,charges,prediction_label
764,45,female,25.175,2,no,northeast,9095.06825,10598.300167
887,36,female,30.020,0,no,northwest,5272.17580,5324.433100
890,64,female,26.885,0,yes,northwest,29330.98315,28849.992486
1293,46,male,25.745,3,no,northwest,9301.89355,10706.753074
259,19,male,31.920,0,yes,northwest,33750.29180,33196.114590


In [73]:
# Step 6: Visualise

# create residual plot  
fig_res = residuals(reg_final.pipeline, reg_exp.X_test, reg_exp.y_test)
fig_res.show()

---
## 4. Save and Deploy Models

In [74]:
# Save classification pipeline
exp.save_model(clf_final.pipeline, "juice_classification_pipeline")
print("Classification model saved as juice_classification_pipeline.pkl")

# Save regression pipeline
reg_exp.save_model(reg_final.pipeline, "insurance_regression_pipeline")
print("Regression model saved as insurance_regression_pipeline.pkl")

Classification model saved as juice_classification_pipeline.pkl
Regression model saved as insurance_regression_pipeline.pkl


In [76]:
# Load and verify classification model
loaded_clf = exp.load_model("juice_classification_pipeline")
clf_loaded_pred = exp.predict_model(loaded_clf)
print("Loaded classification predictions:", clf_loaded_pred.predictions[["Purchase", "prediction_label"]].head(3))

Loaded classification predictions:     Purchase  prediction_label
426       CH                 1
590       CH                 0
573       CH                 1


In [77]:
# Load and verify regression model
loaded_reg = reg_exp.load_model("insurance_regression_pipeline")
reg_loaded_pred = reg_exp.predict_model(loaded_reg)
print("Loaded regression predictions:", reg_loaded_pred.predictions
      [["charges", "prediction_label"]].head(3))

Loaded regression predictions:          charges  prediction_label
764   9095.06825      10598.300167
887   5272.17580       5324.433100
890  29330.98315      28849.992486


---
## Summary

You have completed the full ML workflow:

| Step | What you did |
| ---- | ------------ |
| EDA | Generated interactive profile reports for both datasets |
| Data prep | Cleaned data, split into train/test sets |
| Baseline | Used PyCaret `setup()` to initialise the environment |
| Compare | Ran `compare_models()` to find the best algorithm |
| Tune | Used `tune_model()` to optimise hyperparameters |
| Deploy | Saved pipelines with `save_model()` and verified with `load_model()` |

### Key takeaways
- PyCaret automates the entire modelling loop — setup, compare, tune, deploy.
- Always do EDA first: check target balance, missing values, skewness, and correlations.
- Use a protected holdout set (test data) for final evaluation — never tune on test data.
- Saved pipelines include preprocessing, so you can load and predict on raw data later.

---
<small>© 2026 Decoded Limited. All rights reserved.</small>